# Phase 8 — the tipping point as a critical-length crossing, and the quality of the signal

Pure **re-analysis** of the exact score files that produced `RESULTS.md` (the Colab
A100 draft-surprise run) — **no GPU, no serving, no new generation**. It loads the
phase-4 outputs (`weird_scores.jsonl`, `weird_meta.jsonl`, `baseline_scores.jsonl`)
and adds two things the earlier report did not:

**Part A — the tip as *falling below a critical length*.** Instead of temperature,
the order parameter here is the **length of the chain-of-thought** (the `<think>…</think>`
block). Hypothesis: a transcript tips into the weird behaviour when its think phase is
too short — it *falls below a critical length* `L*`. We locate `</think>` and the tip
per transcript, compare the CoT length of tipped vs non-tipped transcripts, estimate
`L*`, and show the mechanism (the tip fires right where thinking ends).

**Part B — is the signal distributed, accumulating/polar, or noise?** For each
transcript's per-token excess series we compute three *orthogonal* descriptors and
place every transcript on a 2-D map. The key methodological point (verified on
synthetic ground truth in this notebook): **concentration alone cannot tell signal
from noise** — only the temporal permutation test can.

Everything is CPU-only; needs `numpy` + `matplotlib` and the three data files.


In [ ]:
# === Cell 1 — config =========================================================
# Point these at the score files from the RESULTS.md run. On Colab they are on
# your Drive; set MOUNT_DRIVE=True and the paths under DATA_DIR.
MOUNT_DRIVE = True
DATA_DIR    = "/content/drive/MyDrive/weirdspec"      # folder holding the 3 files

WEIRD_SCORES    = "weird_scores.jsonl"        # phase-4 output on the weird set
WEIRD_META      = "weird_meta.jsonl"          # phase-1 metadata sidecar
BASELINE_SCORES = "baseline_scores.jsonl"     # phase-4 output on the held-out null

BEHAVIOR_FOCUS  = "language-switching-english" # the Chinese-onset case study
SIGMA_STRONG    = 5.0     # a token counts as a hard "tip" at +5σ excess vs the null
RUN_SIGMA       = 3.0     # supra-threshold run marker
PERM_N          = 200     # within-transcript permutations for the noise gate
GATE_Z          = 2.5     # temporal-z below this on ALL stats -> classified NOISE
MAX_QUALITY     = None    # cap transcripts in Part B (None = all); set e.g. 3000 if slow
import os
paths = {k: os.path.join(DATA_DIR, v) for k, v in
         dict(weird=WEIRD_SCORES, meta=WEIRD_META, base=BASELINE_SCORES).items()}
print("inputs:", paths)


In [ ]:
# === Cell 2 — load + schema check ===========================================
import json
def read_jsonl(path):
    rows=[]
    with open(path, encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows

if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e:
        print("drive mount skipped:", e)

weird = read_jsonl(paths["weird"])
meta  = read_jsonl(paths["meta"])
base  = read_jsonl(paths["base"])
meta_by_index = dict(enumerate(meta))

need = {"nll_draft","nll_target","token_strs","aggregates"}
have = set(weird[0]) if weird else set()
print(f"weird={len(weird)}  baseline={len(base)}  meta={len(meta)}")
print("fields present:", sorted(have))
missing = need - have
if missing:
    print("!! MISSING", missing, "— this looks like a reduced/compact dump.")
    print("   Part B (numeric) still runs; Part A needs 'token_strs'.")
HAS_STRS = "token_strs" in have
from collections import Counter
bc = Counter(meta_by_index.get(s["index"],{}).get("behavior_id") for s in weird)
print("behaviors:", dict(bc))


In [ ]:
# === Cell 3 — per-token null (same definition as phase 5) ====================
import numpy as np
tok_excess = np.array([nd-nt for s in base for nd,nt in zip(s["nll_draft"], s["nll_target"])], float)
assert tok_excess.size, "baseline has no tokens"
TOK_MEAN = float(tok_excess.mean()); TOK_STD = float(tok_excess.std(ddof=1))
print(f"per-token null excess: {TOK_MEAN:.3f} ± {TOK_STD:.3f}  (n={tok_excess.size})")

def zseries(s):
    """z-scored per-token excess series for one transcript vs the null."""
    e = np.array(s["nll_draft"],float) - np.array(s["nll_target"],float)
    return (e - TOK_MEAN)/TOK_STD


## Part A — the tip as a crossing below a critical CoT length

For each transcript we find three landmarks in the scored-token stream:

* **think-end** — the scored-token index at `</think>` (CoT length `L_cot`).
* **CJK onset** — first token containing a Chinese/Japanese/Korean codepoint
  (the actual language switch — the "Chinese onset" from RESULTS.md).
* **z5 tip** — first token whose excess is ≥ +5σ vs the null (a hard surprise tip).

A transcript is **tipped** if a CJK onset (for language-switching) or a z5 tip exists.
The hypothesis is that tipped transcripts have a *shorter* `L_cot`: the model that
"thinks less" falls below the critical length `L*` and discharges into the weird mode.


In [ ]:
# === Cell 4 — landmarks per transcript ======================================
import unicodedata
def has_cjk(t):
    for ch in t:
        o=ord(ch)
        if 0x3400<=o<=0x9FFF or 0xF900<=o<=0xFAFF or 0x20000<=o<=0x2FA1F: return True
    return False
def thinkend_idx(token_strs):
    """scored-token index at which '</think>' has been emitted, else None."""
    buf=""
    for i,t in enumerate(token_strs):
        buf=(buf+t)[-16:]
        if "</think>" in buf: return i
    return None
def cjk_onset(token_strs):
    for i,t in enumerate(token_strs):
        if has_cjk(t): return i
    return None

rows=[]
for s in weird:
    m=meta_by_index.get(s["index"],{})
    z=zseries(s); n=z.size
    if n==0: continue
    strs = s["token_strs"] if HAS_STRS else None
    te = thinkend_idx(strs) if strs else None
    cj = cjk_onset(strs)   if strs else None
    z5 = next((i for i in range(n) if z[i]>=SIGMA_STRONG), None)
    peak = int(np.argmax(z))
    tip = cj if (m.get("behavior_id")==BEHAVIOR_FOCUS and cj is not None) else z5
    rows.append(dict(index=s["index"], behavior=m.get("behavior_id"),
                     pattern=m.get("pattern_id"), n=n,
                     L_cot=(te+1 if te is not None else None),
                     cjk=cj, z5=z5, peak=peak, tip=tip,
                     tipped=tip is not None,
                     maxz=float(z.max())))
import numpy as np
print(f"{len(rows)} transcripts;  with </think>: {sum(r['L_cot'] is not None for r in rows)};  "
      f"tipped: {sum(r['tipped'] for r in rows)};  CJK onset: {sum(r['cjk'] is not None for r in rows)}")


In [ ]:
# === Cell 5 — critical length: tipped vs non-tipped, L*, mechanism ==========
import numpy as np, matplotlib.pyplot as plt
def subset(behavior=None):
    return [r for r in rows if r["L_cot"] is not None and (behavior is None or r["behavior"]==behavior)]

foc = subset(BEHAVIOR_FOCUS) or subset(None)
Lc = np.array([r["L_cot"] for r in foc]); tp = np.array([r["tipped"] for r in foc])
print(f"focus set: {len(foc)} transcripts, L_cot median={np.median(Lc):.0f} "
      f"[{np.percentile(Lc,10):.0f}, {np.percentile(Lc,90):.0f}]")

# Degeneracy guard: if think blocks barely vary, CoT length is not an order parameter.
if np.percentile(Lc,90) < 3 or Lc.std() < 1.0:
    print("!! think blocks are ~degenerate in this config (thinking was disabled).")
    print("   CoT length cannot act as an order parameter here — to vary it, re-run")
    print("   GENERATION with enable_thinking=True. Part B below still applies.")

if tp.any() and (~tp).any():
    lt, ln = Lc[tp], Lc[~tp]
    print(f"L_cot  tipped: median {np.median(lt):.0f}   non-tipped: median {np.median(ln):.0f}")
    # bootstrap CI on the median difference (non-tipped minus tipped; >0 supports the hypothesis)
    rng=np.random.default_rng(0)
    diffs=[np.median(rng.choice(ln,ln.size))-np.median(rng.choice(lt,lt.size)) for _ in range(2000)]
    lo,hi=np.percentile(diffs,[2.5,97.5])
    print(f"median(non-tipped) - median(tipped) = {np.median(ln)-np.median(lt):.0f}  95% CI [{lo:.0f},{hi:.0f}]")
    # L* via Youden's J on 'tip when L_cot <= L*'
    grid=np.arange(int(Lc.min()), int(Lc.max())+1)
    def youden(L):
        pred = Lc<=L
        tpr = (pred & tp).sum()/max(tp.sum(),1); fpr=(pred & ~tp).sum()/max((~tp).sum(),1)
        return tpr-fpr
    J=[youden(L) for L in grid]; Lstar=int(grid[int(np.argmax(J))])
    print(f"critical length  L* ≈ {Lstar}  (tip when CoT length ≤ L*; Youden J={max(J):.2f})")

    fig,ax=plt.subplots(1,2,figsize=(11,3.8))
    bins=np.linspace(Lc.min(),Lc.max(),25)
    ax[0].hist(ln,bins,alpha=.6,label="non-tipped",density=True)
    ax[0].hist(lt,bins,alpha=.6,label="tipped",density=True)
    ax[0].axvline(Lstar,color="k",ls="--",lw=1); ax[0].set_title("CoT length by outcome"); ax[0].set_xlabel("L_cot"); ax[0].legend()
    # mechanism: where does the tip sit relative to think-end?
    d=[r["tip"]-(r["L_cot"]-1) for r in foc if r["tipped"] and r["tip"] is not None]
    ax[1].hist(d,bins=np.arange(-10,30),color="C2")
    ax[1].axvline(0,color="k",ls="--",lw=1); ax[1].set_title("tip index − think-end (Δ)"); ax[1].set_xlabel("Δ tokens")
    plt.tight_layout(); plt.show()
    med=float(np.median(d)) if d else float("nan")
    if d and abs(med)<=3:
        print(f"Δ (tip − think-end) median = {med:.0f}  → the tip fires right where thinking ends")
    elif d:
        print(f"Δ (tip − think-end) median = {med:.0f}  → tip fires ~{med:.0f} tokens after think-end "
              "(not a clean at-the-boundary discharge in this data)")
else:
    print("not enough tipped/non-tipped split in the focus set to fit L*.")


## Part B — distributed vs accumulating (polar) vs noise

Three orthogonal statistics on each transcript's z-scored excess series, each
answering exactly one question:

1. **Concentration** — participation ratio `PR=(Σw)²/Σw²` on the positive excess
   `w=max(z,0)`, reported as `conc = 1 − PR/n` (0 = spread across all tokens,
   →1 = a single pole). Cross-checked with the Gini coefficient. **Depends only on
   the value histogram → invariant under shuffling**, so it measures *peakedness*,
   i.e. distributed ↔ polar — but says nothing about signal vs noise.
2. **Temporal permutation-z** — CUSUM range, lag-1 autocorrelation and longest
   supra-3σ run, each z-scored against `PERM_N` within-transcript shuffles.
   Shuffling destroys temporal order, so this is the **noise gate**: if all three
   z's are below `GATE_Z`, the structure is no more than the value histogram → **noise**.
   CUSUM also **locates** the accumulation change-point (the pole / tip).
3. **Amplitude** — `maxz` vs the null, so a flat un-elevated transcript can't pass as signal.

Classification: **NOISE** if the gate fails; otherwise **POLAR/accumulating**
(high concentration, extreme localized tokens) vs **DISTRIBUTED** (low concentration
but real temporal structure spread over many tokens).


In [ ]:
# === Cell 6 — signal-quality measures (vectorized) ==========================
import numpy as np
def participation_conc(z):
    w=np.clip(z,0,None); ss=(w*w).sum()
    if ss<=0: return 0.0
    return 1.0 - (w.sum()**2/ss)/len(z)
def gini(z):
    w=np.sort(np.clip(z,0,None)); n=w.size; s=w.sum()
    if s<=0: return 0.0
    return (2*np.sum((np.arange(1,n+1))*w))/(n*s) - (n+1)/n
def _temporal(X):                       # X: (m,n) -> (cusum_range, autocorr, longest_run) per row
    mean=X.mean(1,keepdims=True); S=np.cumsum(X-mean,1)
    cus=S.max(1)-S.min(1)
    d=((X-mean)**2).sum(1); ac=((X[:,:-1]-mean)*(X[:,1:]-mean)).sum(1)/np.where(d>0,d,1)
    B=X>=RUN_SIGMA; best=np.zeros(X.shape[0],int); cur=np.zeros(X.shape[0],int)
    for j in range(X.shape[1]):
        cur=np.where(B[:,j],cur+1,0); best=np.maximum(best,cur)
    return cus,ac,best.astype(float),int(np.argmax(np.abs(S[0]))) if X.shape[0]==1 else None
def quality(z, rng):
    n=z.size
    if n<8: return None
    obs=_temporal(z[None,:]); cp=obs[3]
    P=np.array([rng.permutation(z) for _ in range(PERM_N)])
    pc,pa,pr,_=_temporal(P)
    def zof(o,p): sd=p.std(ddof=1); return float((o-p.mean())/sd) if sd>0 else 0.0
    zc=zof(obs[0][0],pc); za=zof(obs[1][0],pa); zr=zof(obs[2][0],pr)
    gate=max(zc,za,zr)
    conc=participation_conc(z)
    return dict(conc=conc, gini=gini(z), z_cusum=zc, z_ac=za, z_run=zr,
                gate=gate, changepoint=cp, maxz=float(z.max()))

def classify(q):
    if q is None: return "short"
    if q["gate"]<GATE_Z or q["maxz"]<RUN_SIGMA: return "noise"
    return "polar" if (q["conc"]>=0.6 and q["z_run"]>=2.0) else "distributed"
print("measures ready")


In [ ]:
# === Cell 7 — synthetic self-test: does the classifier separate the 3 types? =
import numpy as np
rng=np.random.default_rng(0)
N=200
pole=rng.normal(0,1,N).copy()
for i in range(150,158): pole[i]+=(i-149)*0.6
pole[158]+=12.0                                    # accumulate then discharge
dist=rng.normal(0,1,N).copy(); dist[40:150]+=2.0   # broad elevated plateau
noise=rng.normal(0,1,N)                            # iid
for name,x in [("POLE",pole),("DISTRIBUTED",dist),("NOISE",noise)]:
    q=quality(x, np.random.default_rng(1))
    print(f"{name:12s} conc={q['conc']:.2f} gini={q['gini']:.2f} | "
          f"z_cusum={q['z_cusum']:5.1f} z_ac={q['z_ac']:4.1f} z_run={q['z_run']:4.1f} | "
          f"gate={q['gate']:4.1f} -> {classify(q).upper()}")
print("\nExpected: POLE->POLAR, DISTRIBUTED->DISTRIBUTED, NOISE->NOISE.")
print("Note conc is HIGH for NOISE too — concentration is NOT a noise gate; the")
print("permutation gate is. That is the whole point of using all three measures.")


In [ ]:
# === Cell 8 — classify every transcript + 2-D map + per-behavior table ======
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(7)
work = weird if MAX_QUALITY is None else weird[:MAX_QUALITY]
Q=[]
for s in work:
    q=quality(zseries(s), rng)
    if q is None: continue
    q["behavior"]=meta_by_index.get(s["index"],{}).get("behavior_id"); q["index"]=s["index"]
    q["klass"]=classify(q); Q.append(q)
print(f"classified {len(Q)} transcripts")

from collections import Counter, defaultdict
tab=defaultdict(Counter)
for q in Q: tab[q["behavior"]][q["klass"]]+=1
print(f"\n{'behavior':32s} {'n':>5} {'polar%':>7} {'distr%':>7} {'noise%':>7} {'med conc':>9}")
concby=defaultdict(list)
for q in Q: concby[q["behavior"]].append(q["conc"])
for b,c in sorted(tab.items(), key=lambda kv:-sum(kv[1].values())):
    n=sum(c.values())
    print(f"{str(b)[:32]:32s} {n:5d} {100*c['polar']/n:6.0f}% {100*c['distributed']/n:6.0f}% "
          f"{100*c['noise']/n:6.0f}% {np.median(concby[b]):9.2f}")

# 2-D map: concentration (distributed<->polar) vs temporal gate-z (noise<->signal)
fig,ax=plt.subplots(figsize=(7,5))
behs=sorted({q["behavior"] for q in Q}, key=lambda b:-tab[b].total() if hasattr(tab[b],'total') else 0)
for b in behs:
    pts=[(q["conc"],min(q["gate"],20)) for q in Q if q["behavior"]==b]
    if pts:
        xs,ys=zip(*pts); ax.scatter(xs,ys,s=8,alpha=.4,label=str(b)[:22])
ax.axhspan(0,GATE_Z,color="grey",alpha=.15)          # noise band
ax.axhline(GATE_Z,color="k",lw=.8,ls="--")
ax.axvline(0.6,color="k",lw=.8,ls=":")
ax.text(0.02,GATE_Z*0.35,"NOISE (gate fails)",fontsize=9)
ax.set_xlabel("concentration  (distributed → polar)"); ax.set_ylabel("temporal gate-z  (noise → signal)")
ax.set_title("signal character per transcript"); ax.legend(fontsize=7,markerscale=2,ncol=2)
plt.tight_layout(); plt.show()


In [ ]:
# === Cell 9 — language-switching case study: the Chinese onset ==============
import numpy as np, matplotlib.pyplot as plt
cand=[s for s in weird if meta_by_index.get(s["index"],{}).get("behavior_id")==BEHAVIOR_FOCUS]
if not cand:
    print(f"no '{BEHAVIOR_FOCUS}' transcripts in this file.")
else:
    # pick the transcript with the strongest tip
    best=max(cand, key=lambda s: max(zseries(s)))
    z=zseries(s if False else best); strs=best["token_strs"] if HAS_STRS else None
    te=thinkend_idx(strs) if strs else None; cj=cjk_onset(strs) if strs else None
    peak=int(np.argmax(z))
    print(f"transcript {best['index']}  L_cot={None if te is None else te+1}  CJK onset={cj}  peak@{peak}  maxz={z.max():.1f}")
    if strs:
        marks=[]
        for i,t in enumerate(strs[:200]):
            tt=t.replace("\n","\\n")
            marks.append(f"⟪{tt}⟫" if z[i]>=SIGMA_STRONG else (f"«{tt}»" if z[i]>=RUN_SIGMA else tt))
        print("\n"+ "".join(marks))
    S=np.cumsum(z-z.mean())
    fig,ax=plt.subplots(2,1,figsize=(10,5),sharex=True)
    ax[0].plot(z,lw=.8); ax[0].axhline(SIGMA_STRONG,color="r",ls=":",lw=.8)
    if te is not None: ax[0].axvline(te,color="g",ls="--",lw=1,label="</think>")
    if cj is not None: ax[0].axvline(cj,color="m",ls="--",lw=1,label="CJK onset")
    ax[0].set_ylabel("z excess"); ax[0].legend(fontsize=8)
    ax[1].plot(S,lw=1,color="C3"); ax[1].set_ylabel("CUSUM"); ax[1].set_xlabel("scored token")
    if te is not None: ax[1].axvline(te,color="g",ls="--",lw=1)
    ax[1].set_title("CUSUM: flat through the think phase, kink at the discharge")
    plt.tight_layout(); plt.show()


### Reading the output

* **Part A** answers whether the tip is a *critical-length crossing*: if tipped
  transcripts have a significantly shorter `L_cot` (CI on the median difference
  excludes 0) and `Δ≈0`, the model discharges into the weird mode exactly when the
  think phase runs out — a length threshold `L*`. If the think blocks are degenerate
  (thinking was disabled in the RESULTS.md config), the notebook says so and the
  clean next experiment is to regenerate with `enable_thinking=True`.
* **Part B** answers the *quality* question. The synthetic self-test proves the
  scheme separates the three regimes, and the per-behaviour table + 2-D map show
  which behaviours are **polar/accumulating** (language switching — a single pole
  at the Chinese onset), **distributed**, or **noise**. Crucially the temporal
  permutation gate — not concentration — is what rules out noise.
